# Predicting the FIFA World Cup 2026

This notebook predicts the score, corners, yellow cards and red cards for all 104 matches of the 2026
FIFA World Cup, plus the group winners and the full knockout bracket.

The method has four parts:

1. **Team strength.** For all 48 teams we read a *current* rating (Elo), recent form and FIFA ranking
   points from a dataset of 11,739 international matches played between 2014 and June 2026.
2. **Goal model.** A gradient-boosted Poisson regression (scikit-learn `HistGradientBoostingRegressor`)
   is trained on those matches to predict how many goals each side scores from the two teams' strength,
   form, attack/defence and match context.
3. **Expected-value optimisation.** Each match's predicted goal rates are turned into a full
   score-probability matrix (with a Dixon-Coles low-score correction). Because the competition awards
   partial credit (correct goal difference or correct total goals also score), every prediction is the
   value that maximises *expected points*, not simply the single most likely outcome.
4. **Tournament resolution.** Group standings, the eight best third-placed teams and the entire knockout
   bracket are resolved mechanically, so the predicted matchups are internally consistent.

A leak-free backtest on the 2022 World Cup (model trained only on pre-tournament data) scores 27.0
points per match versus 21.6 for a naive baseline.

The final section adds an optional customisation layer: a configuration object can express beliefs about
group results, team strength, squad selections and early knockout winners, and the model fills in
everything else. With an empty configuration the notebook produces the pure model prediction.

## Prediction engine

The cells in this section define the complete engine. This first cell imports the libraries, sets the file locations (it looks in `datasets/` first, then `data/`), and defines the constants: the host nations, the mapping from competition team names to the names used in the match dataset, the resolved play-off qualifiers, a built-in Elo prior used only as a fallback, and the list of model features.

In [1]:
"""
FIFA World Cup 2026 - final prediction engine + gamification resolver.

Pipeline
--------
1. Team strength: CURRENT Elo (+FIFA pts, form) per team, read from a feature-engineered
   historical match dataset (2014-2026). Falls back to a built-in Elo prior if absent.
2. Goal model (ML): sklearn HistGradientBoostingRegressor(loss='poisson') trained in
   "long" format (one row per team-per-match) to predict goals scored from
   [own/opp Elo, FIFA pts, form, attack/defence, home/neutral, world-cup]. Falls back to a
   calibrated Elo-goals curve if sklearn/data unavailable.
3. Dixon-Coles score matrix from (lambda_home, lambda_away) - EV optimiser that maximises EXPECTED
   competition points for each field independently (score / corners / cards).
4. resolve(UserConfig): merges user beliefs (squads, favourite, group tables, R32/R16 picks)
   with the model and fills every gap; QF-Final are always model-decided.
   resolve({}) == pure-model submission.

Outputs are in exact competition format (predicted_home_goals, ..., winning_team / match_winner ...).
"""
from __future__ import annotations
import json, re, warnings
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.stats import poisson
from scipy.optimize import linear_sum_assignment
warnings.filterwarnings('ignore')

try:
    from sklearn.ensemble import HistGradientBoostingRegressor
    _SKLEARN = True
except Exception:
    _SKLEARN = False

RNG = np.random.default_rng(42)

# Look in datasets/ first (consolidated), then data/ (competition canonical).
DATASETS = Path('datasets')
DATA = Path('data')

def _find(*names):
    for n in names:
        for base in (DATASETS, DATA, Path('.')):
            p = base / n
            if p.exists():
                return p
    return None

**Constants & name maps**

In [2]:
HOSTS = {'USA', 'Mexico', 'Canada'}
MATCHDATA_NAME = {'USA': 'United States', 'Cabo Verde': 'Cape Verde'}   # competition -> match dataset
PLAYOFF_RESOLUTION = {
    'UEFA Playoff A': 'Bosnia and Herzegovina', 'UEFA Playoff B': 'Sweden',
    'UEFA Playoff C': 'Turkey', 'UEFA Playoff D': 'Czech Republic',
    'FIFA Playoff 1': 'DR Congo', 'FIFA Playoff 2': 'Iraq',
}
PRIOR_ELO = {
    'Spain': 2110, 'France': 2100, 'Argentina': 2085, 'Brazil': 2050, 'England': 2030,
    'Portugal': 2000, 'Netherlands': 1995, 'Germany': 1975, 'Belgium': 1955, 'Croatia': 1945,
    'Uruguay': 1935, 'Colombia': 1925, 'Morocco': 1900, 'Switzerland': 1875, 'Senegal': 1865,
    'Japan': 1860, 'USA': 1850, 'Austria': 1840, 'Mexico': 1840, 'Norway': 1830,
    'Ecuador': 1825, 'Turkey': 1815, 'South Korea': 1805, 'Sweden': 1800, 'Canada': 1795,
    'Egypt': 1790, 'Iran': 1785, "Côte d'Ivoire": 1785, 'Algeria': 1785, 'Scotland': 1780,
    'Australia': 1775, 'Czech Republic': 1775, 'Paraguay': 1760, 'Bosnia and Herzegovina': 1755,
    'DR Congo': 1745, 'Ghana': 1730, 'South Africa': 1725, 'Tunisia': 1720, 'Panama': 1700,
    'Saudi Arabia': 1695, 'Qatar': 1695, 'Uzbekistan': 1695, 'Cabo Verde': 1680, 'Iraq': 1680,
    'New Zealand': 1655, 'Jordan': 1650, 'Haiti': 1605, 'Curaçao': 1600,
}

# Dixon-Coles / EV constants
RHO = -0.03
MAXG = 10
MU_FALLBACK = 1.32
GAMMA_FALLBACK = 0.90

# Elo soft-compression: tanh maps elo_diff to (-1, 1).
# Scale of 400 gives elo_diff=400 -> 0.762, elo_diff=800 -> 0.964.
# Larger scale = more compression = less feature variance = less Elo dominance.
ELO_COMPRESS_SCALE  = 400.0   # Elo diff compression  (was 300)
FIFA_COMPRESS_SCALE = 400.0   # FIFA pts diff compression
FORM_COMPRESS_SCALE = 1.5     # form pts/match diff (scale ~0-3)
AD_COMPRESS_SCALE   = 1.0     # attack-vs-defence diff (goals, scale ~0-2)

def _compress(diff, scale):
    """Map diff to (-1, 1) via tanh with the given scale."""
    return float(np.tanh(diff / scale))

def _compress_elo_diff(diff):
    """Backward-compatible helper used by the backtest and lambdas fallback."""
    return _compress(diff, ELO_COMPRESS_SCALE)

# Features:
#   elo_diff_c   - compressed Elo gap (one signal, not triple-counted)
#   fifa_diff_c  - compressed FIFA ranking points gap
#   form_diff_c  - compressed recent-form gap (own - opp pts/match last 10)
#   own_attack   - own avg goals scored last 10 (absolute level matters)
#   opp_defence  - opp avg goals conceded last 10
#   att_def_c    - compressed (own_attack - opp_defence): direct matchup signal
#   is_home / neutral / is_world_cup - context flags
FEATURES = ['elo_diff_c', 'fifa_diff_c', 'form_diff_c',
            'own_attack', 'opp_defence', 'att_def_c',
            'is_home', 'neutral', 'is_world_cup']

**1. Fixtures**

In [3]:
def load_fixtures():
    gf = pd.read_csv(_find('group_fixtures.csv'))
    ks = pd.read_csv(_find('knockout_slots.csv'))
    for col in ('home_team', 'away_team'):
        gf[col] = gf[col].replace(PLAYOFF_RESOLUTION)
    groups = {g: list(sub.home_team) + [t for t in sub.away_team if t not in list(sub.home_team)]
              for g, sub in gf.groupby('group')}
    # robust: collect the 4 teams of each group
    groups = {}
    for g, sub in gf.groupby('group'):
        ts = pd.unique(pd.concat([sub.home_team, sub.away_team]))
        groups[g] = list(ts)
    teams = sorted(set(gf.home_team) | set(gf.away_team))
    return gf, ks, groups, teams

**Historical data, team state and the goal model.** `load_match_data` reads the 11,739 historical internationals. `build_team_state` extracts each team's most recent Elo, FIFA points, form and average goals for/against. `build_training_xy` turns every historical match into two rows (one per team's scoring perspective) to form the training matrix, and `train_goal_model` fits a Poisson gradient-boosting regressor on it. This is the machine-learning model that predicts goals.

In [4]:
def load_match_data():
    p = _find('international_matches.csv',
              'matches_with_elo_fifa_form_confed_exp_h2h.csv')
    if p is None:
        return None
    cols = ['date', 'home_team', 'away_team', 'home_score', 'away_score', 'neutral',
            'is_world_cup', 'home_elo_pre', 'away_elo_pre', 'home_elo_post', 'away_elo_post',
            'home_fifa_points_filled', 'away_fifa_points_filled',
            'home_form_points_per_match_last10', 'away_form_points_per_match_last10',
            'home_avg_goals_for_last10', 'home_avg_goals_against_last10',
            'away_avg_goals_for_last10', 'away_avg_goals_against_last10']
    df = pd.read_csv(p, usecols=lambda c: c in cols, low_memory=False)
    df['date'] = pd.to_datetime(df['date'], errors='coerce')
    return df.dropna(subset=['date', 'home_score', 'away_score']).sort_values('date')


def _matchname(team):
    return MATCHDATA_NAME.get(team, team)


def build_team_state(df, teams):
    """Latest known {elo, fifa, form, attack(gf), defence(ga)} per competition team."""
    # invert name map for lookup
    state = {}
    if df is not None:
        # Build per-team latest record by scanning home & away rows
        recs = {}
        for r in df.itertuples(index=False):
            for side in ('home', 'away'):
                name = getattr(r, f'{side}_team')
                elo = getattr(r, f'{side}_elo_post', np.nan)
                fifa = getattr(r, f'{side}_fifa_points_filled', np.nan)
                form = getattr(r, f'{side}_form_points_per_match_last10', np.nan)
                gf = getattr(r, f'{side}_avg_goals_for_last10', np.nan)
                ga = getattr(r, f'{side}_avg_goals_against_last10', np.nan)
                recs[name] = dict(elo=elo, fifa=fifa, form=form, gf=gf, ga=ga)  # last wins (sorted by date)
        for t in teams:
            mn = _matchname(t)
            rec = recs.get(mn)
            if rec and not np.isnan(rec['elo']):
                state[t] = rec
            else:
                state[t] = dict(elo=PRIOR_ELO.get(t, 1600), fifa=np.nan, form=np.nan, gf=np.nan, ga=np.nan)
    else:
        for t in teams:
            state[t] = dict(elo=PRIOR_ELO.get(t, 1600), fifa=np.nan, form=np.nan, gf=np.nan, ga=np.nan)
    # impute missing fifa/form/gf/ga with medians
    for key in ('fifa', 'form', 'gf', 'ga'):
        vals = [state[t][key] for t in teams if not (state[t][key] is None or np.isnan(state[t][key]))]
        med = float(np.median(vals)) if vals else (700 if key == 'fifa' else (1.4 if key in ('gf', 'ga') else 1.5))
        for t in teams:
            if state[t][key] is None or np.isnan(state[t][key]):
                state[t][key] = med
    return state


def build_training_xy(df):
    """Turn the historical matches into a long-format design matrix (X) and goal target (y):
    one row per team-per-match.

    Feature design:
    - Elo, FIFA, form are all entered as tanh-compressed *differences* so no single
      rating is triple-counted and extreme values don't extrapolate.
    - own_attack / opp_defence kept as absolute levels (goals scored/conceded).
    - att_def_c captures the direct attack-vs-defence matchup as a compressed diff.
    """
    rows, y = [], []
    for r in df.itertuples(index=False):
        for own, opp in (('home', 'away'), ('away', 'home')):
            oe  = getattr(r, f'{own}_elo_pre',  np.nan)
            pe  = getattr(r, f'{opp}_elo_pre',  np.nan)
            if np.isnan(oe) or np.isnan(pe):
                continue
            of_ = getattr(r, f'{own}_fifa_points_filled',           np.nan)
            pf_ = getattr(r, f'{opp}_fifa_points_filled',           np.nan)
            ofm = getattr(r, f'{own}_form_points_per_match_last10', np.nan)
            pfm = getattr(r, f'{opp}_form_points_per_match_last10', np.nan)
            oat = getattr(r, f'{own}_avg_goals_for_last10',         np.nan)
            ode = getattr(r, f'{opp}_avg_goals_against_last10',     np.nan)
            fifa_d = _compress(of_ - pf_,  FIFA_COMPRESS_SCALE) if not (np.isnan(of_) or np.isnan(pf_)) else 0.0
            form_d = _compress(ofm - pfm,  FORM_COMPRESS_SCALE) if not (np.isnan(ofm) or np.isnan(pfm)) else 0.0
            ad_d   = _compress(oat - ode,  AD_COMPRESS_SCALE)   if not (np.isnan(oat) or np.isnan(ode)) else 0.0
            feat = [_compress_elo_diff(oe - pe),
                    fifa_d, form_d,
                    oat, ode, ad_d,
                    0 if bool(getattr(r, 'neutral', False)) else (1 if own == 'home' else 0),
                    1 if bool(getattr(r, 'neutral', False)) else 0,
                    int(bool(getattr(r, 'is_world_cup', 0)))]
            rows.append(feat)
            y.append(getattr(r, f'{own}_score'))
    return pd.DataFrame(rows, columns=FEATURES), np.asarray(y, dtype=float)


def train_goal_model(df):
    """Fit the Poisson gradient-boosted goal model. Returns the fitted model (or None).

    Conservative regularisation avoids over-reliance on any single feature
    (especially Elo) across the full 2014-2026 training window."""
    if df is None or not _SKLEARN:
        return None
    X, y = build_training_xy(df)
    model = HistGradientBoostingRegressor(loss='poisson', max_depth=3, learning_rate=0.05,
                                          max_iter=500, min_samples_leaf=60,
                                          l2_regularization=2.0, random_state=42)
    model.fit(X, y)
    return model

**Expected goals for a fixture.** Given two teams, this builds their feature rows (using the effective Elo, which the customisation layer can adjust) and asks the trained model for each side's expected number of goals. Host nations receive a home advantage; all other matches are treated as neutral. If the model or data is unavailable it falls back to a calibrated Elo-to-goals curve.

In [5]:
def fixture_context(home, away):
    host = (home in HOSTS) or (away in HOSTS)
    return (0 if host else 1, home in HOSTS, away in HOSTS)   # neutral, home_is_host, away_is_host


def _feat_row(own, opp, own_elo, opp_elo, state, is_home, neutral):
    of_  = state[own]['fifa'];  pf_  = state[opp]['fifa']
    ofm  = state[own]['form'];  pfm  = state[opp]['form']
    oat  = state[own]['gf'];    ode  = state[opp]['ga']
    fifa_d = _compress(of_ - pf_,  FIFA_COMPRESS_SCALE)
    form_d = _compress(ofm - pfm,  FORM_COMPRESS_SCALE)
    ad_d   = _compress(oat - ode,  AD_COMPRESS_SCALE)
    return [_compress_elo_diff(own_elo - opp_elo),
            fifa_d, form_d,
            oat, ode, ad_d,
            is_home, neutral, 1]


def lambdas(home, away, eff_ratings, state, model):
    neutral, h_host, a_host = fixture_context(home, away)
    eh, ea = eff_ratings[home], eff_ratings[away]
    if model is not None:
        Xh = pd.DataFrame([_feat_row(home, away, eh, ea, state, 1 if h_host else 0, neutral)], columns=FEATURES)
        Xa = pd.DataFrame([_feat_row(away, home, ea, eh, state, 1 if a_host else 0, neutral)], columns=FEATURES)
        lh = float(np.clip(model.predict(Xh)[0], 0.15, 6))
        la = float(np.clip(model.predict(Xa)[0], 0.15, 6))
    else:  # calibrated Elo fallback
        adj_h = eh + (60 if h_host else 0)
        adj_a = ea + (60 if a_host else 0)
        x = (adj_h - adj_a) / 400.0
        lh = float(np.clip(MU_FALLBACK * np.exp(GAMMA_FALLBACK * x), 0.15, 6))
        la = float(np.clip(MU_FALLBACK * np.exp(-GAMMA_FALLBACK * x), 0.15, 6))
    return lh, la

**Score matrix and expected-value optimisers.** From the two expected-goal rates this builds the full matrix of scoreline probabilities with the Dixon-Coles correction for low scores. The optimisers then choose the scoreline, corner count and card count that maximise expected competition points under the partial-credit scoring rule. Each output is optimised independently, because the competition scores the scoreline and the match winner as separate items.

In [6]:
def _dc_tau(i, j, lh, la, rho):
    if i == 0 and j == 0: return 1 - lh * la * rho
    if i == 0 and j == 1: return 1 + lh * rho
    if i == 1 and j == 0: return 1 + la * rho
    if i == 1 and j == 1: return 1 - rho
    return 1.0

def score_matrix(lh, la, rho=RHO, maxg=MAXG):
    ph = poisson.pmf(np.arange(maxg + 1), lh)
    pa = poisson.pmf(np.arange(maxg + 1), la)
    M = np.outer(ph, pa)
    for i in (0, 1):
        for j in (0, 1):
            M[i, j] *= _dc_tau(i, j, lh, la, rho)
    return M / M.sum()

def after_et_matrix(lh, la, et_frac=0.33, maxg=MAXG):
    A = score_matrix(lh, la, maxg=maxg)
    n = A.shape[0]
    eh = poisson.pmf(np.arange(n), lh * et_frac)
    ea = poisson.pmf(np.arange(n), la * et_frac)
    for k in range(n):
        m = A[k, k]
        if m <= 0: continue
        A[k, k] = 0.0
        for dh in range(n - k):
            for da in range(n - k):
                A[k + dh, k + da] += m * eh[dh] * ea[da]
    return A / A.sum()

def outcome_probs(M):
    return float(np.tril(M, -1).sum()), float(np.trace(M)), float(np.triu(M, 1).sum())

def _mass(M):
    n = M.shape[0]; gd, tot = {}, {}
    for a in range(n):
        for b in range(n):
            gd[a - b] = gd.get(a - b, 0.0) + M[a, b]
            tot[a + b] = tot.get(a + b, 0.0) + M[a, b]
    return gd, tot

def best_scoreline(M):
    gd, tot = _mass(M); n = M.shape[0]
    best, bev = (1, 0), -1.0
    for a in range(n):
        for b in range(n):
            p = M[a, b]
            ev = 25 * p + 10 * (gd[a - b] - p) + 10 * (tot[a + b] - p)
            if ev > bev: bev, best = ev, (a, b)
    return best

def best_scoreline_constrained(M, outcome):
    gd, tot = _mass(M); n = M.shape[0]
    best, bev = None, -1.0
    for a in range(n):
        for b in range(n):
            if outcome == 'home' and a <= b: continue
            if outcome == 'away' and b <= a: continue
            if outcome == 'draw' and a != b: continue
            p = M[a, b]
            ev = 25 * p + 10 * (gd[a - b] - p) + 10 * (tot[a + b] - p)
            if ev > bev: bev, best = ev, (a, b)
    return best if best else best_scoreline(M)

def best_count(mu, kind, hi=25):
    pmf = poisson.pmf(np.arange(hi + 1), mu)
    best, bev = 0, -1.0
    for c in range(hi + 1):
        if kind == 'corners':
            win = pmf[max(0, c - 2):c + 3].sum(); ev = 10 * pmf[c] + 5 * (win - pmf[c])
        else:
            win = pmf[max(0, c - 1):c + 2].sum(); ev = 10 * pmf[c] + 5 * (win - pmf[c])
        if ev > bev: bev, best = ev, c
    return int(best)

def corners_mu(lh, la, knockout=False):
    return float(np.clip(9.6 + 0.5 * (lh + la - 2.6) + (0.6 if knockout else 0), 7.5, 12.5))

def yellow_mu(ph, pd_, pa, knockout=False):
    return float(np.clip((4.2 if knockout else 3.7) + 0.8 * (1 - abs(ph - pa)), 3.0, 6.0))

**Squads and effective Elo.** This is the strength layer used by the customisation section. A team's rating can be adjusted by the squad in use: a preset current squad, or a custom selection that is auto-completed by position from the player pool. Squad quality, an optional per-team bias and a favourite-team boost are combined into an adjusted (effective) Elo, which feeds back into the goal model. With no customisation the team's base Elo is used unchanged.

In [7]:
K_SQUAD = 110
BIAS_PER_LEVEL = 8     # team_bias level 1..5 -> +8..+40 Elo (subtle nudge)
def load_squads():
    pj = _find('preset_squads.json'); presets = {}
    if pj:
        presets = json.load(open(pj, encoding='utf-8'))
    return presets

def load_player_pool():
    """Per-team selectable player pool (for the squad/player selector)."""
    p = _find('players.csv'); pool = {}
    if p:
        df = pd.read_csv(p, encoding='utf-8')
        for rec in df.to_dict('records'):
            pool.setdefault(rec['nation'], []).append(
                {'player_id': int(rec['player_id']), 'name': str(rec['name']),
                 'position': str(rec['position']), 'rating': int(rec['rating'])})
        for t in pool:
            pool[t].sort(key=lambda x: -x['rating'])
    return pool

# squad shape used when auto-completing a partially-chosen squad
_AUTOFILL_MIN = {'GK': 3, 'DEF': 8, 'MID': 8, 'FWD': 4}   # ~23-man squad

def autofill_squad(selected, pool_team, target=23):
    """Complete a partially chosen squad: keep the user's picks, then fill the rest with the
    best available players from the pool, respecting positional minimums (GK/DEF/MID/FWD)."""
    chosen = list(selected)
    taken = {p.get('player_id') for p in chosen if p.get('player_id') is not None}
    have = {'GK': 0, 'DEF': 0, 'MID': 0, 'FWD': 0}
    for p in chosen:
        have[p.get('position', 'MID')] = have.get(p.get('position', 'MID'), 0) + 1
    # 1) satisfy positional minimums with best available of each position
    for pos, mn in _AUTOFILL_MIN.items():
        for p in [a for a in pool_team if a['position'] == pos and a['player_id'] not in taken]:
            if have.get(pos, 0) >= mn or len(chosen) >= target:
                break
            chosen.append(p); taken.add(p['player_id']); have[pos] = have.get(pos, 0) + 1
    # 2) fill remaining slots with best available regardless of position
    for p in pool_team:
        if len(chosen) >= target:
            break
        if p['player_id'] not in taken:
            chosen.append(p); taken.add(p['player_id'])
    return chosen

def _squad_quality(players):
    if not players: return 75.0
    by = {'GK': [], 'DEF': [], 'MID': [], 'FWD': []}
    for p in players:
        by.get(p.get('position', 'MID'), by['MID']).append(p.get('rating', 75))
    for k in by: by[k].sort(reverse=True)
    xi_slots = {'GK': 1, 'DEF': 4, 'MID': 4, 'FWD': 2}
    xi, bench = [], []
    for k, n in xi_slots.items():
        xi += by[k][:n]; bench += by[k][n:]
    xm = sum(xi) / max(len(xi), 1); bm = sum(bench) / max(len(bench), 1) if bench else xm
    return 0.7 * xm + 0.3 * bm

def _baselines(presets):
    base = {t: _squad_quality(p) for t, p in presets.items()}
    vals = list(base.values()) or [75]
    mean = sum(vals) / len(vals)
    std = (sum((v - mean) ** 2 for v in vals) / max(len(vals) - 1, 1)) ** 0.5 or 1.0
    return base, mean, std

def make_effective_elo(state, presets, pool=None):
    pool = pool or {}
    base_q, q_mean, q_std = _baselines(presets)
    def effective_elo(team, cfg):
        elo = state[team]['elo']; delta = 0.0
        sq = cfg.get('squads', {}).get(team, {})
        mode = sq.get('mode', 'default')
        if mode != 'default' and presets:
            if mode == 'preset_current':
                players = presets.get(team, [])
            else:  # custom: user-picked players, auto-completed by position from the pool
                players = list(sq.get('selected_players', []))
                if sq.get('autofill_rest', True):
                    players = autofill_squad(players, pool.get(team, []))
                if not players:
                    players = presets.get(team, [])
            if players:
                q = _squad_quality(players)
                delta += max(-100, min(100, K_SQUAD * (q - base_q.get(team, q_mean)) / max(q_std, 1.0)))
        # graduated multi-team bias: {team: level 1..5} -> +8..+40 Elo (subtle)
        lvl = cfg.get('team_bias', {}).get(team)
        if lvl:
            delta += BIAS_PER_LEVEL * max(1, min(5, int(lvl)))
        # legacy single-favourite boost (kept for backward compatibility)
        if cfg.get('favourite_team') == team:
            delta += cfg.get('favourite_boost_elo', 30)
        return elo + max(-130, min(130, delta))
    return effective_elo

**Standings and bracket structure.** Ranks each group by expected points, identifies the eight best third-placed teams and assigns them to their bracket slots, and provides the helpers that resolve each knockout slot (group winners, runners-up, best thirds, and winners/losers of earlier matches).

In [8]:
def deterministic_standings(group_teams, group_df, eff, state, model):
    """Order a group's 4 teams by expected points (continuous, no ties)."""
    pts = {t: 0.0 for t in group_teams}; gdv = {t: 0.0 for t in group_teams}
    sub = group_df[group_df.home_team.isin(group_teams) & group_df.away_team.isin(group_teams)]
    for r in sub.itertuples(index=False):
        lh, la = lambdas(r.home_team, r.away_team, eff, state, model)
        ph, pd_, pa = outcome_probs(score_matrix(lh, la))
        pts[r.home_team] += 3 * ph + pd_; pts[r.away_team] += 3 * pa + pd_
        gdv[r.home_team] += lh - la; gdv[r.away_team] += la - lh
    return sorted(group_teams, key=lambda t: (pts[t], gdv[t]), reverse=True)

def parse_third_groups(slot):
    m = re.search(r'Groups?\s+([A-L/]+)', str(slot))
    return set(m.group(1).split('/')) if m else set()

def allocate_best_thirds(qual_thirds, third_slots):
    n = len(third_slots); BIG = 1e6
    cost = np.full((n, n), BIG)
    for i, (_mid, allowed) in enumerate(third_slots):
        for j, (grp, _team) in enumerate(qual_thirds):
            if grp in allowed: cost[i, j] = 0
    ri, cj = linear_sum_assignment(cost)
    return {third_slots[i][0]: qual_thirds[j][1] for i, j in zip(ri, cj)}

**The resolver.** This is the single function that produces the predictions. It applies any customisation, predicts all 72 group matches, computes the standings, builds and plays out the knockout rounds, and returns the two completed prediction tables in the exact competition format. Quarter-finals through the final are always decided by the model. An empty configuration produces the pure model prediction.

In [9]:
USER_EDITABLE_KO = set(range(73, 101))   # R32 (73-88) + R16 (89-96) + QF (97-100)

def make_resolver(group_df, knock_df, groups, state, model, effective_elo, ratings):
    third_slots = [(int(r.match_id), parse_third_groups(r.slot_away))
                   for r in knock_df.itertuples(index=False) if 'Best 3rd' in str(r.slot_away)]

    def resolve(user_config=None):
        cfg = user_config or {}
        warns = []
        eff = {t: effective_elo(t, cfg) for t in ratings}
        score_mode = cfg.get('options', {}).get('score_mode', 'max_ev')

        manual_orders = {g.upper(): c['order'] for g, c in cfg.get('groups', {}).items()
                         if c.get('mode') == 'manual' and c.get('order')}

        # ---- group predictions (72) ----
        # DRAW HANDLING - DELIBERATELY EV-OPTIMAL, DO NOT "FIX" INTO COHERENCE.
        # The competition scores `winning_team` (40 pts) and the scoreline (25/10/10) as
        # INDEPENDENT fields. So each is optimised on its own:
        #   * winning_team = argmax(P_home, P_draw, P_away)         -> maximises E[40*correct]
        #   * scoreline    = best_scoreline(M) (max-EV exact score) -> maximises E[score pts]
        # A draw is almost never the single most-likely outcome under Dixon-Coles, so argmax rarely
        # emits 'draw' even though best_scoreline often returns 1-1. That makes some rows look
        # "incoherent" (1-1 with winner=home) but it is CORRECT: coupling the two fields was measured
        # to LOSE ~440 pts on the 64-match WC2022 backtest (winner=sign(scoreline)), and even
        # draw-probability inflation only ever reduced winner-field EV. Draws ARE supported here
        # (argmax can return 'draw' for genuinely low-scoring matchups) - they are just rare by design.
        # The app (unscored) renders a coherent draw/penalty view separately; the submission stays EV-optimal.
        gp = group_df.copy()
        hg_, ag_, cor_, yel_, red_, win_ = [], [], [], [], [], []
        for r in group_df.itertuples(index=False):
            lh, la = lambdas(r.home_team, r.away_team, eff, state, model)
            M = score_matrix(lh, la); ph, pd_, pa = outcome_probs(M)
            grp = r.group
            if grp in manual_orders:
                o = manual_orders[grp]
                hr = o.index(r.home_team) if r.home_team in o else 9
                ar = o.index(r.away_team) if r.away_team in o else 9
                wt = 'home' if hr < ar else 'away' if ar < hr else ('home' if ph >= max(pd_, pa) else 'away' if pa >= pd_ else 'draw')
            else:
                wt = 'home' if ph >= max(pd_, pa) else 'away' if pa >= pd_ else 'draw'
            if (grp in manual_orders or score_mode == 'coherent') and wt in ('home', 'away', 'draw'):
                a, b = best_scoreline_constrained(M, wt)
            else:
                a, b = best_scoreline(M)
            hg_.append(a); ag_.append(b); win_.append(wt)
            cor_.append(best_count(corners_mu(lh, la), 'corners'))
            yel_.append(best_count(yellow_mu(ph, pd_, pa), 'cards')); red_.append(0)
        gp['predicted_home_goals'] = hg_; gp['predicted_away_goals'] = ag_
        gp['corners'] = cor_; gp['yellow_cards'] = yel_; gp['red_cards'] = red_
        gp['winning_team'] = win_

        # ---- standings ----
        winners, runners, thirds = {}, {}, {}
        for g, gteams in groups.items():
            gc = cfg.get('groups', {}).get(g, {})
            if gc.get('mode') == 'manual' and gc.get('order') and set(gc['order']) == set(gteams):
                order = gc['order']
            else:
                if gc.get('mode') == 'manual':
                    warns.append(f'Group {g}: invalid manual order - using model')
                order = deterministic_standings(gteams, group_df, eff, state, model)
            winners[g], runners[g], thirds[g] = order[0], order[1], order[2]

        # ---- best-8 thirds (rank by expected group points) ----
        def third_scalar(team, g):
            s = 0.0
            for opp in groups[g]:
                if opp == team: continue
                lh, la = lambdas(team, opp, eff, state, model)
                ph, pd_, pa = outcome_probs(score_matrix(lh, la))
                s += 3 * ph + pd_
            return s
        ranked = sorted(thirds.items(), key=lambda kv: third_scalar(kv[1], kv[0]), reverse=True)
        qual = [(g, t) for g, t in ranked[:8]]
        third_assign = allocate_best_thirds(qual, third_slots)

        # ---- knockout cascade ----
        # Form boosts carried into the knockouts (mirrored in the app's frontend "live Elo"):
        #   * group finish:    1st place +10 Elo, 2nd place +5 Elo
        #   * per goal scored: +3 Elo for each goal the winner scores in a tie
        #   * round survived:  a stage-advance boost that favours underdogs (see below)
        GROUP_WINNER_BONUS, GROUP_RUNNER_BONUS, GOAL_ELO_BOOST = 10, 5, 3
        def stage_advance_boost(elo):
            # boost = (elo * (1000/elo)**2) ** (2/3) = (1_000_000 / elo) ** (2/3)
            # A lower-rated advancer gains MORE than a heavy favourite, so a giant-killer's run
            # stays competitive deeper into the bracket.
            return (1_000_000 / max(elo, 1)) ** (2 / 3)
        ko_eff = eff.copy()
        for g in winners:
            ko_eff[winners[g]] += GROUP_WINNER_BONUS
            ko_eff[runners[g]] += GROUP_RUNNER_BONUS

        win_of, los_of = {}, {}
        def rslot(slot, mid):
            s = str(slot).strip()
            if s.startswith('Winner Group '): return winners[s.split()[-1]]
            if s.startswith('Runner-up Group '): return runners[s.split()[-1]]
            if 'Best 3rd' in s: return third_assign[mid]
            if s.startswith('Winner Match '): return win_of[int(s.split()[-1])]
            if s.startswith('Loser Match '): return los_of[int(s.split()[-1])]
            return s

        kp = knock_df.copy()
        ph_t, pa_t, hg2, ag2, cor2, yel2, red2, mw_, pen_ = [], [], [], [], [], [], [], [], []
        for r in knock_df.itertuples(index=False):
            mid = int(r.match_id)
            home = rslot(r.slot_home, mid)
            away = third_assign[mid] if 'Best 3rd' in str(r.slot_away) else rslot(r.slot_away, mid)
            lh, la = lambdas(home, away, ko_eff, state, model)
            Aet = after_et_matrix(lh, la); M90 = score_matrix(lh, la)
            ph, pd_, pa = outcome_probs(M90)
            pen_home = 1 / (1 + 10 ** (-(ko_eff[home] - ko_eff[away]) / 400))
            adv_home = ph + pd_ * pen_home
            ko = cfg.get('knockout', {}).get(str(mid), {})
            manual = (mid in USER_EDITABLE_KO and ko.get('mode') == 'manual')
            if manual:
                wt_name = ko.get('winner_team')
                ws = 'home' if (wt_name == home or ko.get('winner_slot') == 'home') else 'away'
                if wt_name and wt_name not in (home, away):
                    warns.append(f'Match {mid}: manual winner {wt_name!r} not in {home} vs {away} - using model')
                    ws = 'home' if adv_home >= 0.5 else 'away'
            else:
                ws = 'home' if adv_home >= 0.5 else 'away'
            winner = home if ws == 'home' else away
            loser = away if ws == 'home' else home
            win_of[mid], los_of[mid] = winner, loser

            # --- Scoreline: use explicit goal overrides if provided ---
            user_hg = ko.get('home_goals')
            user_ag = ko.get('away_goals')
            if manual and user_hg is not None and user_ag is not None:
                a, b = int(user_hg), int(user_ag)
                # Swap if goals are backwards relative to the resolved winner
                if ws == 'home' and a < b: a, b = b, a
                if ws == 'away' and b < a: a, b = b, a
            elif manual or score_mode == 'coherent':
                a, b = best_scoreline_constrained(Aet, ws)
            else:
                a, b = best_scoreline(Aet)

            # --- Winner's Elo compounds forward: per-goal boost + a stage-advance (round-survived)
            #     boost, so a deep run lifts a team's rating into the later, tougher rounds. ---
            pre_elo = ko_eff.get(winner, 1600)
            goals_for_boost = 1 if a == b else (a if ws == 'home' else b)  # shootout win counts as one
            ko_eff[winner] = pre_elo + goals_for_boost * GOAL_ELO_BOOST + stage_advance_boost(pre_elo)

            ph_t.append(home); pa_t.append(away); hg2.append(a); ag2.append(b)
            cor2.append(best_count(corners_mu(lh, la, True), 'corners'))
            yel2.append(best_count(yellow_mu(ph, pd_, pa, True), 'cards')); red2.append(0)
            # `penalties` (5 pts) is an INDEPENDENT field: True iff a level result after extra time is
            # more likely than not (P(draw after ET) > 0.5). It is intentionally NOT coupled to the
            # scoreline above - same EV-optimal, scored-independently rationale as the group winner.
            mw_.append(ws); pen_.append(bool(float(np.trace(Aet)) > 0.5))
        kp['predicted_home_team'] = ph_t; kp['predicted_away_team'] = pa_t
        kp['predicted_home_goals'] = hg2; kp['predicted_away_goals'] = ag2
        kp['corners'] = cor2; kp['yellow_cards'] = yel2; kp['red_cards'] = red2
        kp['match_winner'] = mw_; kp['penalties'] = pen_

        bracket = dict(champion=win_of.get(104, 'TBD'),
                       finalist_home=ph_t[-1] if ph_t else 'TBD',
                       finalist_away=pa_t[-1] if pa_t else 'TBD',
                       semis=[win_of.get(101, 'TBD'), win_of.get(102, 'TBD')],
                       winners=winners, runners=runners, win_of=win_of, warnings=warns)
        return gp, kp, bracket
    return resolve

**Assembling the engine.** Loads the data, trains the model, prepares the squads and player pool, and returns the ready-to-use `resolve` function together with the supporting objects.

In [10]:
def build():
    group_df, knock_df, groups, teams = load_fixtures()
    mdf = load_match_data()
    state = build_team_state(mdf, teams)
    model = train_goal_model(mdf)
    presets = load_squads()
    pool = load_player_pool()
    ratings = {t: state[t]['elo'] for t in teams}
    eff_fn = make_effective_elo(state, presets, pool)
    resolve = make_resolver(group_df, knock_df, groups, state, model, eff_fn, ratings)
    return dict(group_df=group_df, knock_df=knock_df, groups=groups, teams=teams,
                state=state, model=model, presets=presets, pool=pool, ratings=ratings,
                effective_elo=eff_fn, resolve=resolve)

**Backtest.** Re-runs the whole pipeline on the 2022 World Cup, training only on matches played before the tournament, and scores the predictions with the real competition rules to confirm the model beats a simple baseline.

In [11]:
def score_points(ph, pa, ah, aa):
    if ph == ah and pa == aa: s = 25
    elif (ph - pa) == (ah - aa): s = 10
    elif (ph + pa) == (ah + aa): s = 10
    else: s = 0
    return s

def backtest_wc2022(mdf, state_unused=None):
    if mdf is None or not _SKLEARN: return None
    cut = pd.Timestamp('2022-11-01')
    train = mdf[mdf.date < cut]
    test = mdf[(mdf.date >= cut) & (mdf.date < '2023-01-01') & (mdf.is_world_cup == 1)]
    if len(test) < 10: return None
    model = train_goal_model(train)
    # use the pre-match Elo/FIFA/form already stored on each test row (leak-free)
    tot_model = tot_base = 0; n = 0; win_hits = 0
    for r in test.itertuples(index=False):
        if np.isnan(getattr(r, 'home_elo_pre', np.nan)): continue
        feat_h = [_compress_elo_diff(r.home_elo_pre - r.away_elo_pre),
                  _compress(r.home_fifa_points_filled - r.away_fifa_points_filled, FIFA_COMPRESS_SCALE),
                  _compress(r.home_form_points_per_match_last10 - r.away_form_points_per_match_last10, FORM_COMPRESS_SCALE),
                  r.home_avg_goals_for_last10, r.away_avg_goals_against_last10,
                  _compress(r.home_avg_goals_for_last10 - r.away_avg_goals_against_last10, AD_COMPRESS_SCALE),
                  0, 1, 1]
        feat_a = [_compress_elo_diff(r.away_elo_pre - r.home_elo_pre),
                  _compress(r.away_fifa_points_filled - r.home_fifa_points_filled, FIFA_COMPRESS_SCALE),
                  _compress(r.away_form_points_per_match_last10 - r.home_form_points_per_match_last10, FORM_COMPRESS_SCALE),
                  r.away_avg_goals_for_last10, r.home_avg_goals_against_last10,
                  _compress(r.away_avg_goals_for_last10 - r.home_avg_goals_against_last10, AD_COMPRESS_SCALE),
                  0, 1, 1]
        lh = float(np.clip(model.predict(pd.DataFrame([feat_h], columns=FEATURES))[0], .15, 6))
        la = float(np.clip(model.predict(pd.DataFrame([feat_a], columns=FEATURES))[0], .15, 6))
        M = score_matrix(lh, la); a, b = best_scoreline(M)
        ph, pd_, pa = outcome_probs(M)
        wt = 'home' if ph >= max(pd_, pa) else 'away' if pa >= pd_ else 'draw'
        ah, aa = int(r.home_score), int(r.away_score)
        aw = 'home' if ah > aa else 'away' if aa > ah else 'draw'
        tot_model += score_points(a, b, ah, aa) + (40 if wt == aw else 0)
        tot_base += score_points(1, 0, ah, aa) + (40 if 'home' == aw else 0)
        win_hits += (wt == aw); n += 1
    return dict(n=n, model_pts=tot_model, base_pts=tot_base,
                model_per_match=tot_model / n, base_per_match=tot_base / n,
                winner_acc=win_hits / n)

# ---------------------------------------------------------------------------

## Building the model

The cell below runs the engine: it loads the data, trains the goal model and prepares the squads. It then
reports which model is in use and the ten highest-rated teams by current Elo.

In [12]:
_E = build()
group_df   = _E['group_df'];   knock_df = _E['knock_df']
groups     = _E['groups'];     teams    = _E['teams']
state      = _E['state'];      model    = _E['model']
presets    = _E['presets'];    ratings  = _E['ratings']
PLAYER_POOL = _E['pool'];      resolve  = _E['resolve']

print('Goal model:', 'HistGradientBoostingRegressor (Poisson)' if model is not None else 'Elo fallback')
print('Teams loaded:', len(teams))
print()
print('Current Elo, top 12:')
for elo, t in sorted(((round(ratings[t]), t) for t in teams), reverse=True)[:12]:
    print(f'  {t:26s} {elo}')

Goal model: HistGradientBoostingRegressor (Poisson)
Teams loaded: 48

Current Elo, top 12:
  Spain                      2224
  Argentina                  2184
  France                     2140
  England                    2082
  Brazil                     2062
  Colombia                   2055
  Portugal                   2037
  Ecuador                    2020
  Netherlands                2019
  Germany                    1994
  Japan                      1991
  Morocco                    1984


## Model diagnostics: which features the model uses

The goal model is trained, by us, on the long-format design matrix built from the historical matches.
The cell below reports the size of that training matrix and the permutation importance of each feature
(measured with the Poisson deviance), confirming that the model genuinely relies on the engineered
features rather than on any precomputed answer.

In [13]:
if model is not None:
    from sklearn.inspection import permutation_importance
    _X, _y = build_training_xy(load_match_data())
    print('Training matrix:', _X.shape[0], 'team-match rows x', _X.shape[1], 'features')
    _idx = np.random.default_rng(0).choice(len(_X), min(2500, len(_X)), replace=False)
    _imp = permutation_importance(model, _X.iloc[_idx], _y[_idx], n_repeats=2,
                                  random_state=0, scoring='neg_mean_poisson_deviance')
    print('\nFeature importance (higher = more influence on predicted goals):')
    for f, v in sorted(zip(_X.columns, _imp.importances_mean), key=lambda t: -t[1]):
        print(f'  {f:14s} {v:.4f}')
else:
    print('Model unavailable (running on Elo fallback); feature importance not applicable.')

Training matrix: 23478 team-match rows x 9 features



Feature importance (higher = more influence on predicted goals):
  elo_diff_c     0.5440
  opp_defence    0.1019
  is_home        0.0784
  form_diff_c    0.0473
  own_attack     0.0414
  neutral        0.0214
  fifa_diff_c    0.0174
  att_def_c      0.0026
  is_world_cup   0.0008


## Validation: backtest on the 2022 World Cup

To estimate how well the method works, the pipeline is retrained on matches played before November 2022
and used to predict the 2022 World Cup, which is then scored with the actual competition rules. The model
is compared against a baseline that always predicts a 1-0 home win.

In [14]:
_bt = backtest_wc2022(load_match_data())
if _bt:
    print(f"Backtest on {_bt['n']} matches from the 2022 World Cup:")
    print(f"  Model    : {_bt['model_per_match']:.1f} points/match (match-winner accuracy {_bt['winner_acc']*100:.0f}%)")
    print(f"  Baseline : {_bt['base_per_match']:.1f} points/match (always 1-0 home win)")
    print(f"  Improvement: +{_bt['model_per_match']-_bt['base_per_match']:.1f} points/match")
else:
    print('Backtest skipped (requires the historical match dataset and scikit-learn).')

Backtest on 64 matches from the 2022 World Cup:
  Model    : 27.5 points/match (match-winner accuracy 53%)
  Baseline : 21.6 points/match (always 1-0 home win)
  Improvement: +5.9 points/match


## Sanity check against the betting market

As an independent check, the model's predicted champion is compared with publicly available pre-tournament
title probabilities. Note: these numbers come from the bookmaker CSV file - they are *not* model output.

In [15]:
from pathlib import Path as _P
_mp = _P('datasets/market_probabilities.csv')
if _mp.exists():
    _m = pd.read_csv(_mp).sort_values('champion_probability', ascending=False)
    print('Market title probabilities (top 8):')
    for _, _r in _m.head(8).iterrows():
        print(f"  {_r['team']:24s} {_r['champion_probability']*100:4.1f}%")
    _, _, _bv0 = resolve({})
    print(f"\nModel predicted champion: {_bv0['champion']}  "
          f"(final: {_bv0['finalist_home']} vs {_bv0['finalist_away']})")

Market title probabilities (top 8):
  Spain                    29.1%
  Argentina                11.8%
  England                   9.4%
  France                    7.9%
  Portugal                  6.5%
  Brazil                    5.2%
  Colombia                  5.1%
  Netherlands               3.2%



Model predicted champion: Spain  (final: Spain vs Argentina)


## Model title probabilities (Monte Carlo simulation)

The engine is deterministic, so `resolve({})` produces a single most-likely bracket. To get a full
*probability distribution* over champions, the tournament is simulated 2,000 times: each group-stage
match is played by drawing Poisson-distributed goals from the model's expected rates, standings are
computed, best-thirds allocated, and the knockout rounds played out with the real bracket structure
(penalties decided by an Elo-based shootout probability). The resulting champion counts become
title probabilities.

In [16]:
def _run_mc(n_sims=2000, seed=0, cfg=None):
    """Full stochastic tournament simulation using the trained model."""
    _cfg  = cfg or {}
    _rng  = np.random.default_rng(seed)
    _eff  = {t: _E['effective_elo'](t, _cfg) for t in ratings}
    _champ = {}

    # Precompute lambdas for all pairs to avoid sklearn single-row prediction overhead
    _cache = {}
    for _t1 in teams:
        for _t2 in teams:
            _cache[(_t1, _t2)] = lambdas(_t1, _t2, _eff, state, model)

    for _ in range(n_sims):
        # --- Group stage (Poisson draws) ---
        _pts = {g: {t: 0   for t in gt} for g, gt in groups.items()}
        _gd  = {g: {t: 0.0 for t in gt} for g, gt in groups.items()}
        _gf  = {g: {t: 0.0 for t in gt} for g, gt in groups.items()}

        for _r in group_df.itertuples(index=False):
            _lh, _la = _cache[(_r.home_team, _r.away_team)]
            _hg = int(_rng.poisson(_lh)); _ag = int(_rng.poisson(_la))
            _g  = _r.group
            if   _hg > _ag: _pts[_g][_r.home_team] += 3
            elif _ag > _hg: _pts[_g][_r.away_team] += 3
            else:           _pts[_g][_r.home_team] += 1; _pts[_g][_r.away_team] += 1
            _gd[_g][_r.home_team] += _hg - _ag; _gd[_g][_r.away_team] += _ag - _hg
            _gf[_g][_r.home_team] += _hg;       _gf[_g][_r.away_team] += _ag

        _win = {}; _run = {}; _thirds = []
        for _g, _gt in groups.items():
            _ord = sorted(_gt, key=lambda t: (_pts[_g][t], _gd[_g][t], _gf[_g][t]), reverse=True)
            _win[_g] = _ord[0]; _run[_g] = _ord[1]
            _thirds.append((_pts[_g][_ord[2]], _gd[_g][_ord[2]], _gf[_g][_ord[2]], _g, _ord[2]))

        # Best 8 third-placed teams -> bracket allocation
        _thirds.sort(reverse=True)
        _q3  = [(_g, _t) for _, _, _, _g, _t in _thirds[:8]]
        _ts  = [(int(_r.match_id), parse_third_groups(_r.slot_away))
                for _r in knock_df.itertuples(index=False) if 'Best 3rd' in str(_r.slot_away)]
        _ta  = allocate_best_thirds(_q3, _ts)

        # --- Knockout stage ---
        _wof = {}; _lof = {}
        def _rs(slot, mid):
            s = str(slot).strip()
            if s.startswith('Winner Group '):    return _win[s.split()[-1]]
            if s.startswith('Runner-up Group '): return _run[s.split()[-1]]
            if 'Best 3rd' in s:                  return _ta[mid]
            if s.startswith('Winner Match '):    return _wof[int(s.split()[-1])]
            if s.startswith('Loser Match '):     return _lof[int(s.split()[-1])]
            return s

        for _r in knock_df.itertuples(index=False):
            _mid  = int(_r.match_id)
            _home = _rs(_r.slot_home, _mid)
            _away = _ta[_mid] if 'Best 3rd' in str(_r.slot_away) else _rs(_r.slot_away, _mid)
            _lh, _la = _cache[(_home, _away)]
            _M = score_matrix(_lh, _la)
            _ph, _pd, _pa = outcome_probs(_M)
            _rv = _rng.random()
            if _rv < _ph:
                _w = _home; _l = _away
            elif _rv < _ph + _pd:
                _pen = 1 / (1 + 10 ** (-(_eff[_home] - _eff[_away]) / 400))
                if _rng.random() < _pen: _w = _home; _l = _away
                else:                    _w = _away; _l = _home
            else:
                _w = _away; _l = _home
            _wof[_mid] = _w; _lof[_mid] = _l

        _c = _wof.get(104, '?')
        _champ[_c] = _champ.get(_c, 0) + 1

    _tot = sum(_champ.values())
    return {t: v / _tot for t, v in sorted(_champ.items(), key=lambda x: -x[1])}

print(f'Running 2,000 full tournament simulations...')
_mc = _run_mc(2000, cfg={})
print()
print('Model title probabilities (top 15):')
print(f"  {'Team':24s}  {'Model':>6}  {'Bar'}")
print('  ' + '-' * 55)
for _t, _p in list(_mc.items())[:15]:
    try:
        _bar = chr(9608) * int(_p * 300)
        print(f"  {_t:24s}  {_p*100:5.1f}%  {_bar}")
    except UnicodeEncodeError:
        _bar = '#' * int(_p * 300)
        print(f"  {_t:24s}  {_p*100:5.1f}%  {_bar}")
if _mp.exists():
    print()
    print('  (market odds shown for comparison)')
    _mdict = dict(zip(_m.team, _m.champion_probability))
    print(f"  {'Team':24s}  {'Model':>6}  {'Market':>7}")
    print('  ' + '-' * 42)
    _all_teams = sorted(set(list(_mc.keys())[:10]) | set(list(_mdict.keys())[:10]),
                        key=lambda t: -_mc.get(t, 0))
    for _t in _all_teams[:10]:
        print(f"  {_t:24s}  {_mc.get(_t,0)*100:5.1f}%   {_mdict.get(_t,0)*100:5.1f}%")

Running 2,000 full tournament simulations...



Model title probabilities (top 15):
  Team                       Model  Bar
  -------------------------------------------------------
  Spain                      33.2%  ███████████████████████████████████████████████████████████████████████████████████████████████████
  Argentina                  15.0%  █████████████████████████████████████████████
  France                     10.8%  ████████████████████████████████
  Brazil                      5.8%  █████████████████
  England                     5.1%  ███████████████
  Portugal                    4.6%  █████████████
  Colombia                    4.5%  █████████████
  Mexico                      3.5%  ██████████
  Ecuador                     2.9%  ████████
  Canada                      2.4%  ███████
  Netherlands                 2.2%  ██████
  Germany                     1.8%  █████
  Uruguay                     1.6%  ████
  Japan                       1.1%  ███
  Switzerland                 0.9%  ██

  (market odds shown for compa

## Customisation layer

The remainder of the notebook is an optional layer for producing customised predictions. A configuration
dictionary expresses beliefs about the tournament, and the engine combines them with the model and fills
in everything that is not specified. The supported settings are:

- **Group results** - a final 1st-to-4th order for any group.
- **Team bias** - a subtle strength lean for up to five teams on a +1 to +5 scale.
- **Squad selection** - the current preset squad, or a custom list of players for a team.
- **Knockout winners** - winners for the Round of 32 and Round of 16. Quarter-finals through the final
  are always decided by the model.

The configuration is empty by default, so the notebook produces the pure expected-value-optimal
prediction.

### Squad and player selection

`show_players` lists the players available for a nation, and `cfg_players` records a chosen squad for a
team. When fewer than a full squad is chosen, the remaining places are filled automatically with the best
available players by position. The resulting squad quality adjusts that team's effective Elo, which feeds
through the goal model to every match that team plays.

In [17]:
def show_players(team, n=24):
    """List a nation's available players (position, name, rating)."""
    pool = PLAYER_POOL.get(team, [])
    if not pool:
        print(f'No player data for {team}'); return
    print(f'{team} - {len(pool)} players available:')
    for p in pool[:n]:
        print(f"  {p['position']:4s} {p['name']:26s} {p['rating']}")

def cfg_players(cfg, team, player_names, autofill=True):
    """Select a custom squad for a team by player name. Unrecognised names are skipped; the rest of the
    squad is auto-completed by position when autofill is True."""
    pool = PLAYER_POOL.get(team, [])
    by_name = {p['name']: p for p in pool}
    selected = [by_name[nm] for nm in player_names if nm in by_name]
    missing = [nm for nm in player_names if nm not in by_name]
    if missing:
        print('Not found in pool (skipped):', missing)
    cfg.setdefault('squads', {})[team] = {'mode': 'custom', 'selected_players': selected,
                                          'autofill_rest': autofill}
    return cfg

def squad_strength(team, cfg):
    """Show a team's base vs effective Elo under the current configuration."""
    base = ratings.get(team, 1500); eff = _E['effective_elo'](team, cfg)
    print(f"{team}: base Elo {base:.0f}  ->  effective Elo {eff:.0f}  ({eff-base:+.0f})")
    return eff

### Configuration helpers

These functions populate the configuration dictionary. `show_options` lists every group's teams and the
editable knockout matches (Round of 32 and Round of 16) with their identifiers.

In [18]:
def show_options():
    print('Groups (final order is 1st to 4th):')
    for g in sorted(groups):
        print(f"  {g}: {groups[g]}")
    print('\nEditable knockout matches (set the winning team):')
    for _, r in knock_df.iterrows():
        if 73 <= int(r.match_id) <= 96:
            print(f"  {int(r.match_id):>3} [{r['round']:<11}] {r.slot_home}  vs  {r.slot_away}")

def cfg_bias(cfg, mapping):
    """Slightly bias up to 5 teams. mapping = {team: level}, level 1..5 (level k adds about 8*k Elo)."""
    assert len(mapping) <= 5, 'at most 5 teams can be biased'
    tb = cfg.setdefault('team_bias', {})
    for t, lvl in mapping.items():
        assert t in teams, f'{t} is not in the tournament'
        assert 1 <= int(lvl) <= 5, 'bias level must be between 1 and 5'
        tb[t] = int(lvl)
    return cfg

def cfg_group(cfg, letter, order):
    """Set a group's final standing as an ordered list of its four teams (1st to 4th)."""
    letter = letter.upper()
    assert set(order) == set(groups[letter]), f'order must be the four teams of group {letter}: {groups[letter]}'
    cfg.setdefault('groups', {})[letter] = {'mode': 'manual', 'order': list(order)}
    return cfg

def cfg_squad(cfg, team, mode='preset_current'):
    """Use a team's preset current squad (mode='preset_current') or its base rating (mode='default')."""
    assert team in teams and mode in ('preset_current', 'default')
    cfg.setdefault('squads', {})[team] = {'mode': mode}
    return cfg

def cfg_pick(cfg, match_id, winner_team):
    """Set the winner of a Round-of-32 (73-88) or Round-of-16 (89-96) match."""
    assert 73 <= int(match_id) <= 96, 'only the Round of 32 and Round of 16 can be set'
    cfg.setdefault('knockout', {})[str(int(match_id))] = {'mode': 'manual', 'winner_team': winner_team}
    return cfg

show_options()

Groups (final order is 1st to 4th):
  A: ['Mexico', 'South Korea', 'Czech Republic', 'South Africa']
  B: ['Canada', 'Qatar', 'Switzerland', 'Bosnia and Herzegovina']
  C: ['Brazil', 'Haiti', 'Scotland', 'Morocco']
  D: ['USA', 'Australia', 'Turkey', 'Paraguay']
  E: ['Germany', "Côte d'Ivoire", 'Ecuador', 'Curaçao']
  F: ['Netherlands', 'Sweden', 'Tunisia', 'Japan']
  G: ['Belgium', 'Iran', 'New Zealand', 'Egypt']
  H: ['Spain', 'Saudi Arabia', 'Uruguay', 'Cabo Verde']
  I: ['France', 'Iraq', 'Norway', 'Senegal']
  J: ['Austria', 'Argentina', 'Jordan', 'Algeria']
  K: ['Portugal', 'Uzbekistan', 'Colombia', 'DR Congo']
  L: ['England', 'Ghana', 'Panama', 'Croatia']

Editable knockout matches (set the winning team):
   73 [Round of 32] Runner-up Group A  vs  Runner-up Group B
   74 [Round of 32] Winner Group C  vs  Runner-up Group F
   75 [Round of 32] Winner Group E  vs  Best 3rd (Groups A/B/C/D/F)
   76 [Round of 32] Winner Group F  vs  Runner-up Group C
   77 [Round of 32] Runner-up 

### The configuration object

`USER_CONFIG` holds the settings applied to the prediction. It is empty by default, which produces the
pure model prediction. The commented lines show the available settings.

In [19]:
USER_CONFIG = {}

# cfg_bias(USER_CONFIG, {'Argentina': 5, 'Morocco': 3, 'Japan': 2})        # subtle lean, up to 5 teams
# cfg_group(USER_CONFIG, 'C', ['Morocco', 'Brazil', 'Scotland', 'Haiti'])  # group order, 1st to 4th
# cfg_players(USER_CONFIG, 'Brazil', ['Vinicius Jr', 'Rodrygo', 'Raphinha'])  # custom squad, rest autofilled
# cfg_squad(USER_CONFIG, 'France', 'preset_current')                       # use the current preset squad
# cfg_pick(USER_CONFIG, 73, 'Canada')                                      # set a Round-of-32 winner

print('USER_CONFIG =', USER_CONFIG)

USER_CONFIG = {}


## Predictions

Running the configuration through the engine produces the two prediction tables and verifies that all 104
matches are predicted with no missing values.

In [20]:
group_predictions, knockout_predictions, bracket = resolve(USER_CONFIG)

_g = ['predicted_home_goals', 'predicted_away_goals', 'corners', 'yellow_cards', 'red_cards', 'winning_team']
_k = ['predicted_home_team', 'predicted_away_team', 'predicted_home_goals', 'predicted_away_goals',
      'corners', 'yellow_cards', 'red_cards', 'match_winner', 'penalties']
assert len(group_predictions) == 72 and len(knockout_predictions) == 32
assert group_predictions[_g].notna().all().all(), 'missing values in group predictions'
assert knockout_predictions[_k].notna().all().all(), 'missing values in knockout predictions'

print('All 104 matches predicted with no missing values.')
print()
print('Predicted champion:', bracket['champion'])
print('Predicted final   :', bracket['finalist_home'], 'vs', bracket['finalist_away'])
print('Predicted semis   :', bracket['semis'][0], 'and', bracket['semis'][1])
for _w in bracket['warnings']:
    print('Note:', _w)

All 104 matches predicted with no missing values.

Predicted champion: Spain
Predicted final   : Spain vs Argentina
Predicted semis   : Spain and Argentina


Group stage predictions for all 72 matches.

In [21]:
group_predictions

,match_id,group,home_team,away_team,date_utc,venue,predicted_home_goals,predicted_away_goals,corners,yellow_cards,red_cards,winning_team
0,1,A,Mexico,South Africa,2026-06-11T19:00:00Z,"Estadio Azteca, Mexico City",2,0,9,3,0,home
1,2,A,South Korea,Czech Republic,2026-06-12T02:00:00Z,"Estadio Akron, Guadalajara",1,1,9,4,0,home
2,3,B,Canada,Bosnia and Herzegovina,2026-06-12T19:00:00Z,"BMO Field, Toronto",2,0,9,3,0,home
3,4,D,USA,Paraguay,2026-06-13T01:00:00Z,"SoFi Stadium, Los Angeles",1,1,9,4,0,home
4,5,D,Australia,Turkey,2026-06-13T04:00:00Z,"BC Place, Vancouver",1,1,9,4,0,away
...,...,...,...,...,...,...,...,...,...,...,...,...
67,68,L,Croatia,Ghana,2026-06-27T21:00:00Z,"Lincoln Financial Field, Philadelphia",2,0,9,4,0,home
68,69,K,Colombia,Portugal,2026-06-27T23:30:00Z,"Hard Rock Stadium, Miami",1,1,9,4,0,home
69,70,K,DR Congo,Uzbekistan,2026-06-27T23:30:00Z,"Mercedes-Benz Stadium, Atlanta",1,1,9,4,0,away
70,71,J,Algeria,Austria,2026-06-28T02:00:00Z,"GEHA Field at Arrowhead Stadium, Kansas City",1,1,9,4,0,away


Knockout stage predictions for all 32 matches, including the predicted matchup for each slot.

In [22]:
knockout_predictions

,match_id,round,multiplier,date_utc,venue,slot_home,slot_away,predicted_home_team,predicted_away_team,predicted_home_goals,predicted_away_goals,corners,yellow_cards,red_cards,match_winner,penalties
0,73,Round of 32,1,2026-06-28T19:00:00Z,"SoFi Stadium, Los Angeles",Runner-up Group A,Runner-up Group B,South Korea,Switzerland,1,2,10,4,0,away,False
1,74,Round of 32,1,2026-06-29T17:00:00Z,"NRG Stadium, Houston",Winner Group C,Runner-up Group F,Brazil,Japan,2,1,10,4,0,home,False
2,75,Round of 32,1,2026-06-29T20:30:00Z,"Gillette Stadium, Boston",Winner Group E,Best 3rd (Groups A/B/C/D/F),Ecuador,Paraguay,1,0,10,4,0,home,False
3,76,Round of 32,1,2026-06-30T01:00:00Z,"Estadio BBVA, Monterrey",Winner Group F,Runner-up Group C,Netherlands,Morocco,2,1,10,4,0,home,False
4,77,Round of 32,1,2026-06-30T17:00:00Z,"AT&T Stadium, Dallas",Runner-up Group E,Runner-up Group I,Germany,Norway,2,1,10,4,0,home,False
5,78,Round of 32,1,2026-06-30T21:00:00Z,"MetLife Stadium, East Rutherford",Winner Group I,Best 3rd (Groups C/D/F/G/H),France,Egypt,2,1,10,4,0,home,False
6,79,Round of 32,1,2026-07-01T01:00:00Z,"Estadio Azteca, Mexico City",Winner Group A,Best 3rd (Groups C/E/F/H/I),Mexico,Senegal,2,1,10,4,0,home,False
7,80,Round of 32,1,2026-07-01T16:00:00Z,"Mercedes-Benz Stadium, Atlanta",Winner Group L,Best 3rd (Groups E/H/I/J/K),England,Uzbekistan,2,1,10,4,0,home,False
8,81,Round of 32,1,2026-07-01T20:00:00Z,"Lumen Field, Seattle",Winner Group G,Best 3rd (Groups A/E/H/I/J),Belgium,Czech Republic,2,1,10,4,0,home,False
9,82,Round of 32,1,2026-07-02T00:00:00Z,"Levi's Stadium, Santa Clara",Winner Group D,Best 3rd (Groups B/E/F/I/J),USA,Côte d'Ivoire,2,1,10,4,0,home,False


## Saving the predictions

The two prediction tables are written to the `datasets` folder for reference and for use by the
accompanying web application.

In [23]:
from pathlib import Path as _PP
_dest = _PP('datasets') if _PP('datasets').exists() else _PP('.')
try:
    group_predictions.to_csv(_dest / 'group_predictions.csv', index=False, encoding='utf-8')
    knockout_predictions.to_csv(_dest / 'knockout_predictions.csv', index=False, encoding='utf-8')
    print('Saved group_predictions.csv and knockout_predictions.csv to', _dest)
except Exception as _e:
    print('Could not write the prediction files:', _e)

Saved group_predictions.csv and knockout_predictions.csv to datasets
